# MobileNetV2 CIFAR-10 — Baseline Runner

This notebook is intentionally thin: it never contains training logic itself.
It pulls the actual code (`model.py`, `data.py`, `utils.py`, `train.py`) from
the repo, runs it on this Colab GPU, and plots the results.

**Workflow:** edit the `.py` files locally in VS Code -> `git push` -> re-run
the pull cell below -> re-run training. When compression code lands (Q2), it
will be added as new cells at the bottom of this same notebook.

## 0. Setup — run once per session

In [ ]:
# # EDIT THIS: your repo URL
# REPO_URL = "https://github.com/<you>/<repo>.git"
# REPO_DIR = "repo"

# import os
# if not os.path.exists(REPO_DIR):
#     !git clone {REPO_URL} {REPO_DIR}
# else:
#     !cd {REPO_DIR} && git pull

# %cd {REPO_DIR}

In [1]:
# torch/torchvision are preinstalled on Colab; wandb usually isn't
!pip install -q wandb

In [2]:
# Persist checkpoints + data across disconnects
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/mobilenetv2-cifar10/runs/baseline"
import os
os.makedirs(OUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Sanity check — confirm the stride surgery landed correctly
Before committing to a full 150-epoch run, confirm the final feature map is
`[N, 1280, 4, 4]`, not `[N, 1280, 1, 1]` (which would mean the stock 32x
downsampling is still in effect).

In [3]:
import torch
from model.py import MobileNetV2CIFAR

m = build_mobilenetv2_cifar()
x = torch.randn(2, 3, 32, 32)
print("output shape:", m(x).shape)                # expect [2, 10]
print("final feature map:", m.features(x).shape)  # expect [2, 1280, 4, 4]
print(f"params: {sum(p.numel() for p in m.parameters())/1e6:.2f}M")

ModuleNotFoundError: No module named 'model'

## 2. Train the baseline
First run: no `--resume`. If the session disconnects mid-run, re-run this
cell with `--resume {OUT_DIR}/last.pth` appended (uncomment the line below).

In [ ]:
resume_flag = ""
# resume_flag = f"--resume {OUT_DIR}/last.pth"  # uncomment after a disconnect

!python train.py \
  --epochs 150 \
  --batch-size 128 \
  --lr 0.1 \
  --weight-decay 5e-4 \
  --warmup-epochs 5 \
  --label-smoothing 0.1 \
  --width-mult 1.0 \
  --dropout 0.2 \
  --seed 42 \
  --out-dir {OUT_DIR} \
  --use-wandb \
  {resume_flag}

## 3. Plot results — this is the Q1(c) loss/accuracy curve deliverable

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f"{OUT_DIR}/log.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["test_loss"], label="test")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(log["epoch"], log["train_acc1"], label="train")
axes[1].plot(log["epoch"], log["test_acc1"], label="test")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("top-1 accuracy (%)"); axes[1].legend()
axes[1].set_title("Accuracy")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/curves.png", dpi=150)
plt.show()

print(f"Best test top-1: {log['test_acc1'].max():.2f}%")

---
## Next phase: compression (Q2–Q4)
Once `runs/baseline/best.pth` is in hand, the compression module gets its
own cells appended below this line — it will load `best.pth`, apply
configurable weight/activation compression, and log accuracy-vs-compression
sweeps to the same W&B project for the Q3 parallel-coordinates chart.
Nothing above this line needs to change.